In [2]:
import logging
import json
import os
import ast
import sys
import pandas as pd
from dotenv import load_dotenv
from json_repair import repair_json
import faiss
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from nebula3.gclient.net import ConnectionPool
from nebula3.Config import Config
from RAG_pipeline.utils import (
    load_faiss_index, connect_nebula, retrieve_semantic_nodes,
    get_definitions_from_graph, rerank_definitions
)
import logging
import json
import math
import time
from json_repair import repair_json

In [3]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', force=True)

# Load environment variables from .env file
env_path = './.env'
if not os.path.exists(env_path):
    logging.error(f".env file not found at {env_path}. Please create it with your API keys and configuration.")
    sys.exit(1)
load_dotenv(dotenv_path=env_path)

logging.info("Initial setup and environment variables loaded.")

2025-08-13 19:44:20,304 - INFO - Initial setup and environment variables loaded.


In [4]:
_CURR_DIR = os.getcwd()
MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"
INDEX_FILE = os.path.join(_CURR_DIR, "../graph_rag/faiss_index.bin")
TEXTS_FILE = os.path.join(_CURR_DIR, "../graph_rag/semantic_nodes.json")

# --- Debugging & RAG Parameters ---
TASK_NAME = 'reasoning_fct'
PROMPT_ID = 'v3'
MAX_SHOTS = 3
MODEL_NAME_TO_DEBUG = 'deepseek-r1:14b'
QUESTION_ID_TO_DEBUG = "0ac6c5c7-9826-441a-81d5-68478e6299bb"
RETRIEVAL_TOP_K = 30000
RERANK_TOP_K = 15

In [ ]:
def format_shots(shots):
    """Helper function to format few-shot examples."""
    return "\n\n".join([f"Example {i+1}:\n{json.dumps(shot, indent=2)}" for i, shot in enumerate(shots)])




def load_prompt_assets(task_name, prompt_id, max_shots, library_dir="prompt_library"):
    """Loads prompt templates, output formats, and few-shot examples from a library."""
    assets = {"prompt": "", "output_format": "", "shots": []}
    task_dir = os.path.join(library_dir, task_name)

    if not os.path.isdir(library_dir) or not os.path.isdir(task_dir):
        logging.warning(f"Prompt directory not found at {task_dir}. Proceeding without few-shot examples.")
        return assets

    prompts_path = os.path.join(task_dir, "prompts.json")
    if os.path.exists(prompts_path):
        with open(prompts_path, 'r') as f:
            try:
                prompts = json.load(f).get("prompts", [])
                selected = next((p for p in prompts if p.get("id") == prompt_id), None)
                if selected:
                    assets.update(selected)
            except json.JSONDecodeError:
                logging.error(f"Error decoding JSON from {prompts_path}")

    shots_path = os.path.join(task_dir, "shots.json")
    if os.path.exists(shots_path):
        with open(shots_path, 'r') as f:
            try:
                shots_list = json.load(f).get("shots", [])
                loaded_shots = shots_list[0] if shots_list and isinstance(shots_list[0], list) else shots_list
                assets["shots"] = loaded_shots[:max_shots]
            except json.JSONDecodeError:
                logging.error(f"Error decoding JSON from {shots_path}")

    logging.info(f"Loaded {len(assets['shots'])} shots for task '{task_name}' using prompt '{prompt_id}'.")
    return assets

In [6]:

try:
    st_model = SentenceTransformer(MODEL_NAME)
    faiss_index, faiss_texts = load_faiss_index()
    nebula_client, nebula_pool = connect_nebula()
    llm_client = OpenAI(base_url=os.getenv("OPENAI_BASE_URL"), api_key=os.getenv("API_KEY"))
    logging.info("Successfully loaded models, FAISS index, and connected to NebulaGraph.")
except Exception as e:
    logging.error(f"Failed to initialize models or clients: {e}")
    sys.exit(1)

data_file = f"data/{TASK_NAME}.csv"
if not os.path.exists(data_file):
    logging.error(f"Data file not found at {data_file}. Please ensure the path is correct.")
    sys.exit(1)

df = pd.read_csv(data_file)
question_rows = df[df['id'] == QUESTION_ID_TO_DEBUG]

if question_rows.empty:
    logging.error(f"Question ID '{QUESTION_ID_TO_DEBUG}' not found in {data_file}.")
    sys.exit(1)

question_row = question_rows.iloc[0]

try:
    question = question_row['question']
    options = ast.literal_eval(question_row['options'])
except (KeyError, SyntaxError) as e:
    logging.error(f"Failed to parse question data for ID '{QUESTION_ID_TO_DEBUG}': {e}")
    sys.exit(1)

print(f"\n--- DEBUGGING ID: {QUESTION_ID_TO_DEBUG} ---")
print(f"Question: {question}")
print(f"Options: {options}")
print("--------------------------------------------------\n")

2025-08-13 19:46:47,140 - INFO - Use pytorch device_name: cuda:0
2025-08-13 19:46:47,141 - INFO - Load pretrained SentenceTransformer: pritamdeka/S-PubMedBert-MS-MARCO
2025-08-13 19:47:01,253 - INFO - FAISS index loaded with 7316480 vectors.
2025-08-13 19:47:01,260 - INFO - Get connection to ('127.0.0.1', 9669)
2025-08-13 19:47:01,265 - INFO - Successfully connected to NebulaGraph.
2025-08-13 19:47:01,505 - INFO - Successfully loaded models, FAISS index, and connected to NebulaGraph.



--- DEBUGGING ID: 0ac6c5c7-9826-441a-81d5-68478e6299bb ---
Question: A 56-year-old man with hypertension, diabetes, renal function creatinine 1.6 mg / dL, pre-treatment blood pressure was 170/100 mmHg. This long-term control of blood pressure in patients with the best goals Why?
Options: {'0': '＜130/80 mmHg', '1': '150-160/90-95 mmHg', '2': '＜140/90 mmHg', '3': '＜140/85 mmHg', 'correct answer': '150-160/90-95 mmHg'}
--------------------------------------------------



In [7]:
no_rag_flag = False
final_context = []

if not no_rag_flag:
    logging.info(f"--- STAGE 1: Semantic Retrieval (Top {RETRIEVAL_TOP_K}) ---")
    query = f"{question} {' '.join(options.values())}"
    suis, top_semantic_texts = retrieve_semantic_nodes(query, st_model, faiss_index, faiss_texts, top_k=RETRIEVAL_TOP_K, top_m=30)
    logging.info(f"Retrieved {len(top_semantic_texts)} semantic context documents.")
    print("--------------------------------------------------\n")

    logging.info("--- STAGE 2: Knowledge Graph Traversal ---")
    graph_definitions = get_definitions_from_graph(nebula_client, suis)
    logging.info(f"Retrieved {len(graph_definitions)} definitions from the graph.")
    print("--------------------------------------------------\n")

    logging.info(f"--- STAGE 3: Re-ranking (Top {RERANK_TOP_K}) ---")
    final_definitions = rerank_definitions(question, graph_definitions, top_k=RERANK_TOP_K)
    logging.info(f"Re-ranked to the top {len(final_definitions)} most relevant context documents.")
    final_context = list(set(top_semantic_texts + final_definitions))
    logging.info(f"Combined semantic and graph contexts into {len(final_context)} unique documents.")
else:
    logging.info("--- STAGE 3: RAG Pipeline Skipped ---")

2025-08-13 19:47:29,124 - INFO - --- STAGE 1: Semantic Retrieval (Top 30000) ---
Batches:   0%|          | 0/1 [00:00<?, ?it/s]/home/macharya/dev/medkg-eval/medkg/lib/python3.13/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.49it/s]
2025-08-13 19:47:30,527 - INFO - Retrieved 30000 SUIs and the top 30 semantic texts.
2025-08-13 19:47:30,528 - INFO - Retrieved 30 semantic context documents.
2025-08-13 19:47:30,529 - INFO - --- STAGE 2: Knowledge Graph Traversal ---


--------------------------------------------------



2025-08-13 19:47:31,079 - INFO - Retrieved 217 definitions from the graph.
2025-08-13 19:47:31,080 - INFO - --- STAGE 3: Re-ranking (Top 15) ---


--------------------------------------------------



Some weights of BertForSequenceClassification were not initialized from the model checkpoint at pritamdeka/S-PubMedBert-MS-MARCO and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2025-08-13 19:47:32,058 - INFO - Use pytorch device: cuda:0
Batches: 100%|██████████| 7/7 [00:00<00:00, 21.49it/s]
2025-08-13 19:47:32,683 - INFO - Re-ranked 217 definitions and selected the top 15.
2025-08-13 19:47:32,684 - INFO - Re-ranked to the top 15 most relevant context documents.
2025-08-13 19:47:32,685 - INFO - Combined semantic and graph contexts into 45 unique documents.


In [8]:
prompt_assets = load_prompt_assets(TASK_NAME, PROMPT_ID, MAX_SHOTS, library_dir="prompt_library")
prompt_assets

2025-08-13 19:48:22,890 - INFO - Loaded 3 shots for task 'reasoning_fct' using prompt 'v3'.


{'prompt': "You are a highly intelligent and accurate medical domain expert and a teacher. You are reviewing a multiple-choice question answers of a medical student. You are given questions, options, and answers provided by the colleague.There is a possibility that the student's answer could be wrong. Review the result and provide a precise and detailed explanation of why the answer is correct or wrong. Additionally, you also provide why the other options are not correct. Ensure that the explanation is detailed and accurate. Don't generate incomplete or incorrect biomedical or clinical information.",
 'output_format': "Your output format is valid JSON format {'is_answer_correct': 'yes/no', 'cop_index': 'The integer index of the correct option.', 'answer': 'The full string value of the correct option.', 'why_correct': 'A detailed explanation in list format for the correct answer. This list MUST follow a specific three-part structure: 1. Briefly state the key concepts in the question and

In [18]:
def generate_llm_response(
    llm_client,
    model_name,
    question,
    options,
    context, # Renamed from 'definitions' for clarity
    prompt_assets,
    no_rag=False,
    get_logprobs_for_field: str = None,
    target_tokens: list = None
):
    """
    Generates a response from the LLM, with an added mode for calculating log probabilities.

    Args:
        context (list): The final combined list of context strings.
        get_logprobs_for_field (str, optional): The JSON key for which to calculate probabilities.
        target_tokens (list, optional): The expected string values for the target field (e.g., ['yes', 'no']).
    """
    main_prompt_instruction = prompt_assets.get("prompt", "")
    few_shot_str = format_shots(prompt_assets.get("shots", []))
    # Note: Ensure options are formatted as a string, not with escaped newlines.
    options_str = "\n".join([f"{k}: {v}" for k, v in options.items()])
    
    context_block = ""
    if not no_rag:
        context_str = " ".join(context) if context else "No relevant biomedical context found."
        context_block = f"Context: {context_str}\n\n"
    #print(prompt_assets.get('output_format', ''))
    base_prompt = (
        f"{main_prompt_instruction}\n"
        f"output_format: {prompt_assets.get('output_format', '')}\n\n"
        f"Examples:\n{few_shot_str}\n\n"
        f"--- CURRENT TASK ---\n"
        f"{context_block}"
        f"Question: {question}\nOptions:\n{options_str}\n\n"
    )

    # --- MODE 1: LOGPROBABILITY CALCULATION ---
    if get_logprobs_for_field and target_tokens:
        # Construct a prompt that ends right before the value we want to predict.
        # This forces the model to predict 'yes' or 'no' as the very next token.
        output_format_str = prompt_assets.get("output_format", "")
        
        # The marker is the key followed by a colon and the opening quote for the value.
        split_marker = f"'{get_logprobs_for_field}': '"
        #print(split_marker)
        if split_marker not in output_format_str:
            logging.error(f"Could not find marker '{split_marker}' in output_format asset.")
            return None
        
        # Take the JSON structure up to the point of prediction
        prompt_ending = output_format_str.split(split_marker)[0] + split_marker
        final_prompt = base_prompt + f"Provide your answer. Your output must start with the following JSON structure:\n{prompt_ending}"
        #print(final_prompt)

        try:
            response = llm_client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": final_prompt}],
                temperature=0.0,
                max_tokens=5,  # We only need the next token
                logprobs=True,  # Request log probabilities
                top_logprobs=10 # Get top 10 to ensure our targets are included
            )
            return response
            top_logprobs = response.choices[0].logprobs.content[0].top_logprobs
            
            # Find the log probabilities for our specific target tokens
            token_logprobs = {item.token.lower().strip(' "'): item.logprob for item in top_logprobs}

            # Calculate the raw probabilities, defaulting to -infinity if not found
            raw_probs = {
                token: math.exp(token_logprobs.get(token, -math.inf)) for token in target_tokens
            }

            # Normalize the probabilities between our choices so they sum to 1.0
            total_prob = sum(raw_probs.values())
            if total_prob == 0:
                logging.warning(f"None of the target tokens {target_tokens} were in the top {len(top_logprobs)} logprobs.")
                return {token: 0.0 for token in target_tokens}

            normalized_distribution = {token: prob / total_prob for token, prob in raw_probs.items()}
            return normalized_distribution

        except Exception as e:
            logging.error(f"API call for logprobs failed: {e}")
            return None

    # --- MODE 2: FULL RESPONSE GENERATION (Original behavior) ---
    else:
        final_prompt = base_prompt + f"Provide your answer. You MUST provide your response as a single, valid JSON object with the keys specified in the output_format."
        for attempt in range(2):
            try:
                response = llm_client.chat.completions.create(
                    model=model_name,
                    messages=[{"role": "user", "content": final_prompt}],
                    temperature=0.0,
                    response_format={"type": "json_object"}
                )
                raw_text = response.choices[0].message.content
                # Use json_repair for robustness
                return json.loads(repair_json(raw_text))
            except Exception as e:
                logging.warning(f"Attempt {attempt + 1} failed: {e}")
                if attempt == 0: time.sleep(1) # Wait before retrying
        
        logging.error("Failed to get valid LLM response after multiple attempts.")
        return None

In [19]:
#probability_distribution 
response = generate_llm_response(
    llm_client=llm_client,
    model_name=MODEL_NAME_TO_DEBUG,
    question=question,
    options=options,
    context=final_context,
    prompt_assets=prompt_assets,
    no_rag=no_rag_flag,
    get_logprobs_for_field="is_answer_correct", # <-- Set the target field
    target_tokens=['yes', 'no']                 # <-- Set the expected outcomes
)

2025-08-13 20:50:58,261 - INFO - HTTP Request: POST https://ollama.zib.de/api/chat/completions "HTTP/1.1 200 OK"


In [21]:
response.choices

[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="<think>\nOkay, I'm trying to figure out why the correct answer is option 1: 150-160/90-95 mmHg for this patient's blood pressure control goals. Let me break down what I know.\n\nThe patient is a 56-year-old man with hypertension and diabetes, which are both chronic conditions that can affect kidney function. His creatinine level is 1.6 mg/dL, which I remember isn't extremely high but still indicates some degree of renal impairment because normal levels are usually lower than 1.2 or so.\n\nHe had a pre-treatment blood pressure of 170/100 mmHg. That's quite elevated—stage 2 hypertension for sure. Now, the question is about his long-term BP control goals. I'm trying to recall the target BP levels for patients with multiple comorbidities and especially those with kidney issues.\n\nI think that for people without kidney problems or diabetes, the target is less than 130/80 mmHg. But when there are ot